# Animer — train collaborative-filtering embeddings

**Goal:** train item embeddings from real user-rating behavior on the Kaggle MAL dataset, then upload them to your Supabase `cf_embedding` column.

Why this matters: synopsis embeddings only know what an anime is *about*. CF embeddings know what it *feels like to watch*, because they're learned from how millions of real users rate things together.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU** (not strictly needed but speeds up SVD)
2. Have your Supabase URL + service_role key ready
3. Have a Kaggle account (free): https://www.kaggle.com/account → API → "Create New Token" → downloads `kaggle.json`
4. Run `schema_cf.sql` in your Supabase SQL Editor first (adds the `cf_embedding` column)

Total time: ~10-20 min depending on dataset size.

## 1. Install dependencies

In [ ]:
!pip install -q kagglehub implicit supabase pandas numpy scipy tqdm

## 2. Credentials

Paste your Supabase values. Kaggle credentials will be requested by kagglehub on first download.

In [ ]:
import os

os.environ['SUPABASE_URL'] = 'https://xxxxx.supabase.co'   # ← REPLACE
os.environ['SUPABASE_SERVICE_KEY'] = 'eyJ...'              # ← REPLACE

assert os.environ['SUPABASE_URL'].startswith('https://'), 'SUPABASE_URL not set'
print('✓ Supabase creds set')

## 3. Download Kaggle MAL ratings

We use the popular "Anime Recommendations Database" by CooperUnion — ~7M ratings from ~73k users on ~12k anime. It's smaller than the azathoth42 set but easier to work with, fits comfortably in Colab RAM, and trains in minutes. We can upgrade later.

On first run, kagglehub will prompt you to upload your `kaggle.json` (or set credentials via env vars).

In [ ]:
import kagglehub
import pandas as pd

# Download dataset (cached after first time)
path = kagglehub.dataset_download('CooperUnion/anime-recommendations-database')
print('Dataset files:', os.listdir(path))

ratings = pd.read_csv(f'{path}/rating.csv')
anime_meta = pd.read_csv(f'{path}/anime.csv')

print(f'\nRatings:  {len(ratings):,} rows')
print(f'Anime:    {len(anime_meta):,} rows')
print(f'Users:    {ratings.user_id.nunique():,}')
print(f'\nRating range: {ratings.rating.min()} to {ratings.rating.max()}')
ratings.head()

## 4. Filter + clean ratings

- Drop `rating = -1` (means "watched, did not rate")
- Drop very-low-activity users (≤ 5 ratings) — pure noise
- Drop very-cold anime (≤ 10 ratings)

In [ ]:
df = ratings[ratings.rating >= 1].copy()
print(f'After dropping -1 ratings: {len(df):,}')

# Iterative cleaning — drop low-activity users/anime until stable
for _ in range(3):
    u_counts = df.groupby('user_id').size()
    a_counts = df.groupby('anime_id').size()
    df = df[df.user_id.isin(u_counts[u_counts >= 5].index)]
    df = df[df.anime_id.isin(a_counts[a_counts >= 10].index)]

print(f'After filtering: {len(df):,} ratings, {df.user_id.nunique():,} users, {df.anime_id.nunique():,} anime')

## 5. Build sparse user-item matrix

We treat the rating as a confidence weight (centered around 6.5 so high ratings are positive, low ratings are negative).

In [ ]:
from scipy.sparse import csr_matrix
import numpy as np

# Re-index users and anime to contiguous integers
user_idx = {u: i for i, u in enumerate(df.user_id.unique())}
anime_idx = {a: i for i, a in enumerate(df.anime_id.unique())}
idx_to_anime = {i: a for a, i in anime_idx.items()}

rows = df.user_id.map(user_idx).values
cols = df.anime_id.map(anime_idx).values

# Confidence = score - 5.5 (centered: 10→4.5, 1→-4.5)
# Implicit ALS expects positive confidence; we keep only positive interactions
# and weight high ratings more strongly.
scores = df.rating.values.astype(np.float32)
confidence = np.maximum(scores - 5.5, 0.5)  # floor at 0.5 so 1-5 ratings still count weakly

matrix = csr_matrix((confidence, (rows, cols)),
                    shape=(len(user_idx), len(anime_idx)))

print(f'Matrix: {matrix.shape}, density: {matrix.nnz / (matrix.shape[0]*matrix.shape[1]) * 100:.3f}%')

## 6. Train ALS

ALS (Alternating Least Squares) learns user and item embeddings such that user·item dot product ≈ observed confidence. Item embeddings are what we want — anime with similar embeddings are co-rated by similar people.

Factors = 64 (matches our schema's `vector(64)`). 20 iterations is plenty for this dataset.

In [ ]:
import implicit
from implicit.als import AlternatingLeastSquares

# implicit expects the user-item matrix
model = AlternatingLeastSquares(
    factors=64,
    regularization=0.05,
    iterations=20,
    use_gpu=False,  # set True if Colab GPU is available; CPU is fast enough here
)

model.fit(matrix)

print(f'\n✓ Trained. Item embeddings shape: {model.item_factors.shape}')

## 7. Sanity check — nearest neighbors of a known anime

Toradora! has MAL ID 4224 in this dataset. Let's see what it considers similar.

In [ ]:
TORADORA_MAL = 4224

if TORADORA_MAL in anime_idx:
    tidx = anime_idx[TORADORA_MAL]
    sim_ids, sim_scores = model.similar_items(tidx, N=11)
    print(f'Anime similar to Toradora! (MAL {TORADORA_MAL}):\n')
    for i, s in zip(sim_ids[1:], sim_scores[1:]):  # skip itself
        mal_id = idx_to_anime[i]
        meta = anime_meta[anime_meta.anime_id == mal_id]
        title = meta.name.iloc[0] if len(meta) else '?'
        print(f'  {s:.3f}  [MAL {mal_id}]  {title}')
else:
    print('Toradora not in filtered dataset — try a different anime.')

## 8. Match anime IDs to your Supabase table

Pull all anime in Supabase that have a `mal_id`, then write the CF embedding for each match.

In [ ]:
from supabase import create_client

sb = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_KEY'])

# Fetch all anime with mal_id (paginated to avoid 1k row limit)
all_rows = []
page = 0
PAGE = 1000
while True:
    res = sb.table('anime').select('id, mal_id').not_.is_('mal_id', 'null') \
        .range(page*PAGE, (page+1)*PAGE - 1).execute()
    if not res.data: break
    all_rows.extend(res.data)
    if len(res.data) < PAGE: break
    page += 1

print(f'Supabase has {len(all_rows)} anime with mal_id')

# Map: supabase_id → CF embedding (only for anime that exist in the trained model)
supabase_to_emb = {}
missing = 0
for row in all_rows:
    mid = row['mal_id']
    if mid in anime_idx:
        emb = model.item_factors[anime_idx[mid]]
        supabase_to_emb[row['id']] = emb
    else:
        missing += 1

print(f'Matched: {len(supabase_to_emb)} / {len(all_rows)}  (missing from Kaggle: {missing})')

## 9. Upload CF embeddings to Supabase

We update existing rows — no schema changes (the `cf_embedding` column was created by `schema_cf.sql`).

In [ ]:
from tqdm import tqdm

BATCH = 200
items = list(supabase_to_emb.items())
failed = []

for i in tqdm(range(0, len(items), BATCH), desc='Uploading'):
    batch = items[i:i+BATCH]
    rows = [
        {'id': sb_id, 'cf_embedding': emb.tolist()}
        for sb_id, emb in batch
    ]
    try:
        sb.table('anime').upsert(rows, on_conflict='id', returning='minimal').execute()
    except Exception as e:
        print(f'\nBatch {i} failed: {e}, retrying per-row')
        for r in rows:
            try:
                sb.table('anime').upsert([r], on_conflict='id', returning='minimal').execute()
            except Exception as e2:
                failed.append(r['id'])

print(f'\n✓ Done. Failed: {len(failed)}')

# Final sanity check
verify = sb.table('anime').select('id', count='exact') \
    .not_.is_('cf_embedding', 'null').execute()
print(f'Anime with cf_embedding in Supabase: {verify.count}')